# CodePause: Phase 1C Colab-Only QLoRA Pivot
Official execution environment for CodePause Phase 1.

In [ ]:
import torch
import sys

if not torch.cuda.is_available():
    print("\u274c CRITICAL ERROR: CUDA is not available. Please enable GPU in Runtime settings.")
    sys.exit(1)

device_name = torch.cuda.get_device_name(0)
print(f"Detected GPU: {device_name}")

if "T4" not in device_name:
    print(f"\u274c CRITICAL ERROR: Expected T4 GPU, but got {device_name}. This notebook is optimized for T4.")
    sys.exit(1)

print("\u2705 T4 GPU detected. Proceeding...")

In [ ]:
import os
from google.colab import drive

DRIVE_ROOT = '/content/drive/MyDrive'
if not os.path.isdir(DRIVE_ROOT):
    try:
        drive.mount('/content/drive', force_remount=True, timeout_ms=120000)
    except ValueError as exc:
        raise RuntimeError(
            'HARD STOP: Google Drive mount failed. In Colab, open the Files panel, '
            'disconnect/reconnect Drive or restart only the runtime, then rerun this block.'
        ) from exc

if not os.path.isdir(DRIVE_ROOT):
    raise RuntimeError(f'HARD STOP: Google Drive is not available at {DRIVE_ROOT}')

DRIVE_PATH = f'{DRIVE_ROOT}/codepause_phase1c_outputs'
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"✅ Google Drive mounted and output directory verified: {DRIVE_PATH}")

In [ ]:
%%bash
set -euo pipefail
export PIP_DISABLE_PIP_VERSION_CHECK=1
export PIP_PROGRESS_BAR=off

echo '[1/6] Clone or update repo'
if [ ! -d "codepause/.git" ]; then
    rm -rf codepause
    git clone --depth 1 https://github.com/alesierraalta/AMD-Hacktaton-thinking-middle.git codepause
else
    git -C codepause fetch --depth 1 origin main
    git -C codepause reset --hard origin/main
fi
cd codepause

echo '[2/6] Install requirements'
python -m pip install -q -r requirements.txt

echo '[3/6] Refresh bitsandbytes'
python -m pip uninstall -y -q bitsandbytes || true
python -m pip install -q --no-cache-dir bitsandbytes

echo '[4/6] Show key package versions'
python - <<'PY'
import importlib.metadata as md
for pkg in ['torch', 'transformers', 'trl', 'peft', 'bitsandbytes']:
    try:
        print(f'{pkg}=={md.version(pkg)}')
    except md.PackageNotFoundError:
        print(f'{pkg}=NOT_INSTALLED')
PY

echo '[5/6] Run fast validation tests'
python -m pytest -q tests/test_training_sft_lora.py tests/test_validate_dataset.py

echo '[6/6] Environment setup complete'

In [ ]:
import os, subprocess, sys, time

def run_step(name, cmd, cwd='codepause'):
    print(f'\n=== {name} - START ===', flush=True)
    print(' '.join(cmd), flush=True)
    start = time.time()
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    rc = proc.wait()
    elapsed = time.time() - start
    print(f'=== {name} - END rc={rc} elapsed={elapsed:.1f}s ===', flush=True)
    if rc != 0:
        raise RuntimeError(f'{name} failed with exit code {rc}')

print('[Qwen 0.5B][START]', flush=True)
print('cwd before:', os.getcwd(), flush=True)
os.makedirs('codepause/results', exist_ok=True)

run_step('GPU status', ['nvidia-smi'], cwd='.')

run_step('Qwen 0.5B train adapter', [
    sys.executable, '-u', 'training/sft_lora.py',
    '--model_name', 'Qwen/Qwen2.5-Coder-0.5B-Instruct',
    '--dataset_path', 'data/thinkanywhere_sft.jsonl',
    '--output_dir', 'results/sft_0.5B',
    '--epochs', '1',
    '--max_steps', '10',
    '--max_seq_length', '512',
    '--limit_samples', '8',
    '--per_device_train_batch_size', '1',
    '--load_in_4bit',
])

adapter_config = 'codepause/results/sft_0.5B/adapter_config.json'
if not os.path.exists(adapter_config):
    raise FileNotFoundError(adapter_config)
print('[Qwen 0.5B] Adapter artifact verified:', adapter_config, flush=True)
print(os.listdir('codepause/results/sft_0.5B'), flush=True)

run_step('Qwen 0.5B evaluate baseline', [
    sys.executable, '-u', 'eval/evaluate_baseline.py',
    '--model_name', 'Qwen/Qwen2.5-Coder-0.5B-Instruct',
    '--problems_path', 'data/problems.jsonl',
    '--output_path', 'results/baseline_0.5B.jsonl',
    '--timeout', '10',
])

baseline_path = 'codepause/results/baseline_0.5B.jsonl'
if not os.path.exists(baseline_path):
    raise FileNotFoundError(baseline_path)
print('[Qwen 0.5B] Baseline lines:', sum(1 for _ in open(baseline_path)), flush=True)

run_step('Qwen 0.5B evaluate fine-tuned adapter', [
    sys.executable, '-u', 'eval/evaluate_finetuned.py',
    '--base_model', 'Qwen/Qwen2.5-Coder-0.5B-Instruct',
    '--adapter_path', 'results/sft_0.5B',
    '--problems_path', 'data/problems.jsonl',
    '--output_path', 'results/finetuned_0.5B.jsonl',
    '--timeout', '10',
])

finetuned_path = 'codepause/results/finetuned_0.5B.jsonl'
if not os.path.exists(finetuned_path):
    raise FileNotFoundError(finetuned_path)
print('[Qwen 0.5B] Fine-tuned lines:', sum(1 for _ in open(finetuned_path)), flush=True)
print('[Qwen 0.5B] Complete', flush=True)


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd codepause
mkdir -p results

echo '[Qwen 1.5B][1/5] Train 4-bit LoRA adapter'
python -u training/sft_lora.py \
  --model_name Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --dataset_path data/thinkanywhere_sft.jsonl \
  --output_dir results/sft_1.5B \
  --epochs 1 \
  --max_steps 10 \
  --max_seq_length 1024 \
  --limit_samples 8 \
  --per_device_train_batch_size 1 \
  --load_in_4bit
test -f results/sft_1.5B/adapter_config.json
echo '[Qwen 1.5B][2/5] Adapter saved'

echo '[Qwen 1.5B][3/5] Evaluate baseline'
python -u eval/evaluate_baseline.py \
  --model_name Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --problems_path data/problems.jsonl \
  --output_path results/baseline_1.5B.jsonl \
  --timeout 10
test -f results/baseline_1.5B.jsonl

echo '[Qwen 1.5B][4/5] Evaluate fine-tuned adapter'
python -u eval/evaluate_finetuned.py \
  --base_model Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --adapter_path results/sft_1.5B \
  --problems_path data/problems.jsonl \
  --output_path results/finetuned_1.5B.jsonl \
  --timeout 10
test -f results/finetuned_1.5B.jsonl

echo '[Qwen 1.5B][5/5] Generate report'
python -u eval/make_report.py --input_path results/finetuned_1.5B.jsonl
echo '[Qwen 1.5B] Complete'

In [ ]:
%%bash
cp -r codepause/results/* /content/drive/MyDrive/codepause_phase1c_outputs/
echo "\u2705 All outputs copied to Google Drive!"